# Нейроюрист по сделкам с жилой недвижимостью — прототип (Этап № 3)

**Тема.** Нейро-сотрудник — юрист-консультант для физических лиц по сделкам с **жилой** недвижимостью:
купля-продажа, наём, ипотека, участие в долевом строительстве (ДДУ), дарение, рента и пожизненное
содержание с иждивением, наследование жилья, налоги при продаже/дарении/наследовании, государственная
регистрация прав.

**Задача.** По бытовому вопросу пользователя дать корректную консультацию **со ссылками на конкретные
нормы с реквизитами** (статьи ГК/ЖК/СК/НК, номера федеральных законов, реквизиты постановлений
Пленума ВС РФ и судебных актов), не выходя за периметр жилой недвижимости и не придумывая нормы,
которых нет в базе.

**Алгоритм (RAG + многошаговая цепочка LLM).**
1. Разбор вопроса и сборка поисковых запросов (вызов LLM № 1).
2. Поиск фрагментов в векторной базе FAISS + генерация ответа со ссылками (вызов LLM № 2).
3. Проверка ответа на соответствие найденным источникам, при необходимости — один повтор генерации (вызов LLM № 3).

Плюс — ведение истории диалога (уточняющие вопросы).


## 1. Установка зависимостей

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu openai tiktoken

## 2. База знаний и её обработка

База знаний собрана вручную из открытых официальных источников (ГАРАНT, КонсультантПлюс, портал ВС РФ,
справочник постатейной практики gkrfkod.ru) и хранится в отдельном GitHub-репозитории — тексты
законов и судебных актов скачаны/скопированы из первоисточника и извлечены локально (`pypdf` / `python-docx`),
**без автоматического парсинга сайтов и без прогона через языковую модель**: для юридического RAG точность
цитирования нормы критична.

Состав (~5.4 МБ, 87 текстовых файлов):

| Каталог | Файлов | Что внутри |
|---|---|---|
| `laws/` | 25 | ГК РФ (гл. 30, 32–35, 61–64 + точечные статьи), ЖК РФ (гл. 5–6), СК РФ (гл. 7–8), ФЗ-218, ФЗ-102, ФЗ-214, НК РФ (ст. 217/217.1/220) |
| `plenum/` | 10 | Постановления Пленума ВС РФ по темам периметра |
| `practice/` | 48 | Судебная практика по конкретным статьям ГК (до 5 актов ВС РФ на статью) |
| `obzory/` | 4 | Точечные пункты обзоров практики Президиума ВС РФ |

Конфиденциальных / персональных данных в базе нет — только опубликованные нормативные акты и
обезличенная судебная практика ВС РФ.

Обработка в ноутбуке: загрузка текстов → присвоение каждому документу **реквизитов** (для корректных
ссылок в ответе) → разбиение на чанки с сохранением метаданных → построение эмбеддингов
(`text-embedding-3-small`) → индекс FAISS.


In [ ]:
# Клонируем репозиторий с базой знаний.
REPO_URL = "https://github.com/vaar970-cmyk/neurojurist-realestate.git"
REPO_DIR = "neurojurist-realestate"

import os, shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 {REPO_URL} {REPO_DIR}

KB_DIR = os.path.join(REPO_DIR, "knowledge-base")
TESTSET_PATH = os.path.join(REPO_DIR, "testset", "test_questions.md")
print("Каталоги базы знаний:", sorted(os.listdir(KB_DIR)))

### 2.1. Ключ OpenAI

Ключ берётся из «секретов» Colab (значок 🔑 слева, имя `OPENAI_API_KEY`, тумблер «Notebook access» включён).
Если секрет не задан — ноутбук спросит ключ через безопасный ввод.


In [ ]:
import os, getpass

def setup_api_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
    if not key:
        key = getpass.getpass("Введите OPENAI_API_KEY: ")
    os.environ["OPENAI_API_KEY"] = key

setup_api_key()
print("Ключ установлен:", bool(os.environ.get("OPENAI_API_KEY")))

### 2.2. Реквизиты документов

Чтобы нейроюрист ссылался на источники корректно («Постановление Пленума ВС РФ от 29.05.2012 № 9», а
не просто «постановление Пленума»), каждому файлу базы сопоставляются полные реквизиты. Для законов —
это норма и её место в кодексе, для постановлений и обзоров — дата, номер и название, для постатейной
практики — указание, что реквизиты конкретных судебных актов находятся в тексте фрагмента.


In [ ]:
# Реквизиты законов и подзаконных актов (ключ — имя файла).
SOURCE_META = {
    # --- laws ---
    "fz_102_ipoteka.txt": "Федеральный закон от 16.07.1998 № 102-ФЗ «Об ипотеке (залоге недвижимости)»",
    "fz_214_ddu.txt": "Федеральный закон от 30.12.2004 № 214-ФЗ «Об участии в долевом строительстве многоквартирных домов и иных объектов недвижимости»",
    "fz_218_registraciya.txt": "Федеральный закон от 13.07.2015 № 218-ФЗ «О государственной регистрации недвижимости»",
    "gk_forma_sdelok.txt": "ГК РФ, ст. 158–165.1 (форма сделок, нотариальное удостоверение, государственная регистрация)",
    "gk_glava30_kachestvo_tovara.txt": "ГК РФ, ст. 469–478 (качество товара, последствия передачи товара с недостатками)",
    "gk_glava34_dopolnitelnye_stati.txt": "ГК РФ, ст. 609, 611, 612, 619, 620, 622 (аренда)",
    "gk_glava_30_kuplya_prodazha.txt": "ГК РФ, глава 30 (ст. 454–566, купля-продажа, в т. ч. § 7 продажа недвижимости)",
    "gk_glava_32_darenie.txt": "ГК РФ, глава 32 (ст. 572–582, дарение)",
    "gk_glava_33_renta.txt": "ГК РФ, глава 33 (ст. 583–605, рента и пожизненное содержание с иждивением)",
    "gk_glava_34_arenda.txt": "ГК РФ, глава 34 (ст. 606–670, аренда)",
    "gk_glava_35_naem_zhilya.txt": "ГК РФ, глава 35 (ст. 671–688, наём жилого помещения)",
    "gk_glava_61_65_nasledovanie.txt": "ГК РФ, главы 61–64 (ст. 1110–1175, наследование)",
    "gk_iskovaya_davnost.txt": "ГК РФ, глава 12 (ст. 195–208, исковая давность)",
    "gk_nedeystvitelnost_sdelok.txt": "ГК РФ, ст. 166–181 (недействительность сделок)",
    "gk_neosnovatelnoe_obogaschenie.txt": "ГК РФ, глава 60 (ст. 1102–1109, неосновательное обогащение)",
    "gk_predstavitelstvo_doverennost.txt": "ГК РФ, глава 10 (ст. 182–189, представительство и доверенность)",
    "gk_st209_soderzhanie_prava_sobstvennosti.txt": "ГК РФ, ст. 209 (содержание права собственности)",
    "gk_st292_prava_chlenov_semyi.txt": "ГК РФ, ст. 292 (права членов семьи собственника жилого помещения)",
    "gk_ubytki_neustoyka_zadatok.txt": "ГК РФ, ст. 15, 329–333, 380–381, 393–396 (убытки, неустойка, задаток, проценты)",
    "gk_vozniknovenie_prava_sobstvennosti.txt": "ГК РФ, ст. 218–234 (основания приобретения права собственности, в т. ч. ст. 223 — момент возникновения права у приобретателя)",
    "gk_zaschita_prava_sobstvennosti.txt": "ГК РФ, ст. 301–306 (защита права собственности, виндикация, добросовестный приобретатель)",
    "nk_rf_ndfl_prodazha.txt": "НК РФ, ст. 217 (п. 17.1, 18, 18.1), ст. 217.1 (освобождение от НДФЛ при продаже недвижимости), ст. 220 (имущественные вычеты)",
    "sk_glava_7_8_imuschestvo_suprugov.txt": "СК РФ, ст. 33–44 (законный и договорный режим имущества супругов, в т. ч. ст. 35 — согласие супруга на сделку)",
    "zhk_glava_5_prava_sobstvennika.txt": "ЖК РФ, ст. 30–32 (права и обязанности собственника жилого помещения и членов его семьи)",
    "zhk_glava_6_obschee_imuschestvo.txt": "ЖК РФ, ст. 36–48 (общее имущество собственников помещений в многоквартирном доме)",
    # --- plenum ---
    "postanovlenie_9_2012.txt": "Постановление Пленума ВС РФ от 29.05.2012 № 9 «О судебной практике по делам о наследовании»",
    "postanovlenie_14_2009.txt": "Постановление Пленума ВС РФ от 02.07.2009 № 14 «О некоторых вопросах, возникших в судебной практике при применении Жилищного кодекса Российской Федерации»",
    "postanovlenie_17_2012.txt": "Постановление Пленума ВС РФ от 28.06.2012 № 17 «О рассмотрении судами гражданских дел по спорам о защите прав потребителей»",
    "postanovlenie_23_2023.txt": "Постановление Пленума ВС РФ от 27.06.2023 № 23 «О применении судами правил о залоге вещей»",
    "postanovlenie_25_2015.txt": "Постановление Пленума ВС РФ от 23.06.2015 № 25 «О применении судами некоторых положений раздела I части первой ГК РФ»",
    "postanovlenie_43_2015.txt": "Постановление Пленума ВС РФ от 29.09.2015 № 43 «О некоторых вопросах, связанных с применением норм ГК РФ об исковой давности»",
    "postanovlenie_49_2018.txt": "Постановление Пленума ВС РФ от 25.12.2018 № 49 «О некоторых вопросах применения общих положений ГК РФ о заключении и толковании договора»",
    "postanovlenie_10_22_2010.txt": "Постановление Пленума ВС РФ № 10 и Пленума ВАС РФ № 22 от 29.04.2010 «О некоторых вопросах, возникающих в судебной практике при разрешении споров, связанных с защитой права собственности и других вещных прав»",
    "postanovlenie_73_vas_2011.txt": "Постановление Пленума ВАС РФ от 17.11.2011 № 73 «Об отдельных вопросах практики применения правил ГК РФ о договоре аренды»",
    "obzor_ddu_2017.txt": "Обзор практики разрешения судами споров, возникающих в связи с участием граждан в долевом строительстве, утв. Президиумом ВС РФ 19.07.2017",
    # --- obzory ---
    "obzor_2015_istrebovanie_zhilya.txt": "Обзор судебной практики по делам, связанным с истребованием жилых помещений от граждан по искам государственных органов и органов местного самоуправления, утв. Президиумом ВС РФ 25.11.2015",
    "obzor_1_2022_nasledovanie.txt": "Обзор судебной практики ВС РФ № 1 (2022), утв. Президиумом ВС РФ 01.06.2022",
    "obzor_2_2025_dobrosovestny_priobretatel.txt": "Обзор судебной практики ВС РФ № 2 (2025), утв. Президиумом ВС РФ 18.06.2025",
    "obzor_3_2025_kuplya_prodazha_egrn.txt": "Обзор судебной практики ВС РФ № 3 (2025), утв. Президиумом ВС РФ 08.10.2025",
}

import re

def citation_for(fname):
    if fname in SOURCE_META:
        return SOURCE_META[fname]
    m = re.match(r"st_(\d+)_praktika\.txt", fname)          # постатейная практика
    if m:
        return (f"Судебная практика по ст. {m.group(1)} ГК РФ "
                f"(подборка актов ВС РФ; реквизиты каждого акта — в тексте фрагмента)")
    return fname

print("Реквизитов в справочнике:", len(SOURCE_META))

### 2.3. Загрузка текстов базы знаний

Каждый файл превращается в `Document` с метаданными:
* `source` — имя файла;
* `category` — раздел базы (`laws` / `plenum` / `practice` / `obzory`);
* `citation` — реквизиты документа для ссылки в ответе.


In [ ]:
import glob
from langchain_core.documents import Document

CATEGORY_TITLES = {
    "laws": "законодательство",
    "plenum": "постановление Пленума ВС РФ",
    "practice": "судебная практика по статье ГК РФ",
    "obzory": "обзор практики Президиума ВС РФ",
}

def load_knowledge_base(kb_dir):
    docs = []
    for category in CATEGORY_TITLES:
        for path in sorted(glob.glob(os.path.join(kb_dir, category, "*.txt"))):
            text = open(path, encoding="utf-8").read().strip()
            if not text:
                continue
            fname = os.path.basename(path)
            docs.append(Document(
                page_content=text,
                metadata={"source": fname, "category": category, "citation": citation_for(fname)},
            ))
    return docs

raw_docs = load_knowledge_base(KB_DIR)

from collections import Counter
by_cat = Counter(d.metadata["category"] for d in raw_docs)
total_chars = sum(len(d.page_content) for d in raw_docs)
print(f"Загружено документов: {len(raw_docs)}")
for cat, n in by_cat.items():
    print(f"  {cat:9s}: {n} файлов")
print(f"Суммарный объём текста: {total_chars/1_000_000:.2f} млн символов\n")
print("Пример реквизитов:")
for d in raw_docs[:2] + raw_docs[-15:-13]:
    print(f"  {d.metadata['source']:38s} -> {d.metadata['citation']}")

### 2.4. Разбиение на чанки

Используем `RecursiveCharacterTextSplitter`. Список разделителей начинается с `«\nСтатья »` и
`«\nПостановление »` — так сплиттер старается резать текст по границам статей и судебных актов,
а не посреди нормы. Метаданные исходного документа (включая реквизиты) наследуются каждым чанком.

`chunk_size` и `chunk_overlap` — **гиперпараметры, влияние которых на качество поиска исследуется в
экспериментальной части диплома (Этап № 4).** Здесь взяты рабочие значения по умолчанию.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\nСтатья ", "\nПостановление ", "\nОпределение ", "\n\n", "\n", ". ", " ", ""],
    keep_separator=True,
)

chunks = splitter.split_documents(raw_docs)
print(f"Чанков всего: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c.page_content) for c in chunks)//len(chunks)} символов")
for cat, n in Counter(c.metadata["category"] for c in chunks).items():
    print(f"  {cat:9s}: {n} чанков")
print("\nПример чанка:\n" + "-"*60)
print(chunks[0].page_content[:400], "...")
print("метаданные:", chunks[0].metadata)

### 2.5. Векторный индекс FAISS

Эмбеддинги — `text-embedding-3-small` (OpenAI). Индекс строится один раз и сохраняется локально,
чтобы при повторном запуске нижних ячеек не пересчитывать эмбеддинги.


In [ ]:
import time
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

INDEX_DIR = "faiss_index_v2"   # версия метаданных базы (реквизиты)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if os.path.isdir(INDEX_DIR):
    knowledge_base = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
    print("Индекс загружен с диска.")
else:
    t0 = time.time()
    knowledge_base = FAISS.from_documents(chunks, embeddings)
    knowledge_base.save_local(INDEX_DIR)
    print(f"Индекс построен за {time.time()-t0:.1f} с и сохранён в {INDEX_DIR}/")

print("Векторов в индексе:", knowledge_base.index.ntotal)

### 2.6. Проверка поиска (sanity-check)

In [ ]:
for q in ["переход права собственности на квартиру после подписания договора",
          "наниматель квартиры при смене собственника жилого помещения"]:
    print(f"\nЗапрос: {q}")
    for d in knowledge_base.similarity_search(q, k=3):
        head = d.page_content.strip().split("\n", 1)[0][:70]
        cit = d.metadata.get('citation') or citation_for(d.metadata['source'])
        print(f"  • {cit[:70]}\n      ({head})")

## 3. Структура нейро-сотрудника

```
                 ┌───────────────────────────────────────────────────────────┐
   вопрос  ─────▶ │  ШАГ 1. Разбор вопроса и сборка поисковых запросов (LLM)  │
 (+ история,      │  бытовой вопрос → юридические подвопросы + точные запросы  │
  + тема меню)    │  подсказки по терминологии (наём ≠ аренда, ст. 223/551…)   │
                 │  выход: JSON {themes: [...], queries: [...]}               │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ queries[]
                                             ▼
                 ┌───────────────────────────────────────────────────────────┐
                 │  ШАГ 2. Поиск в FAISS + генерация ответа (LLM)            │
                 │  по каждому запросу: k фрагментов + отдельно k_laws       │
                 │  фрагментов только из законов → объединение уникальных →  │
                 │  ответ со ссылками на нормы и реквизиты документов        │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ черновик ответа + контекст
                                             ▼
                 ┌───────────────────────────────────────────────────────────┐
                 │  ШАГ 3. Проверка ответа (LLM)                            │
                 │  сверка ссылок и утверждений с найденными фрагментами     │
                 │  JSON {ok: bool, problems: [...]}                         │
                 │  если ok=false → один повтор ШАГА 2 с учётом замечаний    │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ финальный ответ
                                             ▼
                                        пользователь
                                             │
                          история диалога ◀──┘  (вопрос + ответ добавляются в буфер,
                                                 используются на ШАГЕ 1 следующего вопроса)
```

**Блоки:**
* **Шаг 1 — разбор и сборка запроса.** Один вызов LLM. Переводит бытовую формулировку в юридические
  категории и разбивает составной вопрос на подвопросы. Содержит подсказки по терминологии, чтобы
  поиск не уходил в смежный институт (наём жилья ≠ аренда). Тема из меню бота — мягкая подсказка,
  **не** фильтр поиска.
* **Шаг 2 — поиск и генерация.** Для каждого подзапроса — векторный поиск по всей базе **плюс**
  отдельный поиск только по разделу `laws` (чтобы в контексте всегда был текст самой нормы, а не
  только практика). Генерация ответа с обязательными ссылками на статьи и реквизиты документов.
* **Шаг 3 — проверка.** Отдельный вызов LLM сверяет черновик с контекстом: нет ли ссылок на нормы
  вне найденных фрагментов и утверждений без опоры на источник. При существенных проблемах —
  **один** повтор генерации (не бесконечный цикл).
* **История диалога.** Буфер «вопрос → ответ», подаётся на Шаг 1 и Шаг 2 для обработки уточнений.


## 4. Реализация алгоритма

Модель — `gpt-4o-mini`, `temperature=0` (для юридического ассистента нужна воспроизводимость и
минимум «фантазии»). Используется клиент `openai` напрямую.


In [ ]:
import json
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

def chat(system, user, temperature=0):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    return resp.choices[0].message.content

def parse_json_block(text):
    """Достаёт JSON из ответа модели (на случай ```json ... ``` обёртки)."""
    t = text.strip()
    if t.startswith("```"):
        t = t.strip("`")
        t = t[4:].strip() if t.lower().startswith("json") else t
    start, end = t.find("{"), t.rfind("}")
    return json.loads(t[start:end + 1])

### Шаг 1. Разбор вопроса и сборка поисковых запросов

In [ ]:
THEMES = ["купля-продажа", "наём/аренда", "ипотека", "ДДУ", "налоги",
          "регистрация", "дарение", "рента", "наследование", "общее"]

STEP1_SYSTEM = f"""Ты — ассистент юриста по сделкам физических лиц с ЖИЛОЙ недвижимостью.
Твоя задача — разобрать бытовой вопрос пользователя и подготовить запросы для поиска по базе
законов и судебной практики.

Сделай следующее:
1. Определи тему(ы) вопроса из списка: {THEMES}.
2. Если вопрос составной — раздели его на самостоятельные подвопросы.
3. Для каждого подвопроса сформулируй короткий юридически точный поисковый запрос
   (термины закона, а не бытовые слова).
4. Учитывай историю диалога: вопрос может быть уточнением к предыдущему.

Подсказки по терминологии (используй правильные термины в запросах):
- Если жильё снимает ГРАЖДАНИН для проживания — это НАЁМ жилого помещения (глава 35 ГК РФ,
  ст. 671–688), наймодатель и наниматель. НЕ аренда. Аренда (глава 34 ГК РФ) — когда помещение
  снимает юридическое лицо. Пример: «хозяин продал квартиру, которую я снимаю» → запрос про
  сохранение договора НАЙМА при смене собственника жилого помещения.
- Момент перехода права собственности на недвижимость к покупателю — государственная регистрация
  в ЕГРН (ст. 223, 551 ГК РФ).
- Освобождение от НДФЛ при продаже жилья зависит от минимального срока владения — ст. 217.1 НК РФ.
- Согласие супруга на распоряжение общим имуществом — ст. 35 СК РФ.

Верни СТРОГО JSON без пояснений:
{{"themes": ["..."], "queries": ["...", "..."]}}"""

def step1_build_queries(question, history="", menu_theme=None):
    hint = f"\nПодсказка от пользователя (тема из меню): {menu_theme}" if menu_theme else ""
    user = f"История диалога:\n{history or '(пусто)'}{hint}\n\nВопрос пользователя:\n{question}"
    raw = chat(STEP1_SYSTEM, user)
    try:
        data = parse_json_block(raw)
        queries = [q for q in data.get("queries", []) if q.strip()]
        themes = data.get("themes", [])
    except Exception:
        queries, themes = [], []
    if not queries:                      # страховка, если модель не вернула JSON
        queries = [question]
    return themes, queries

### Шаг 2. Поиск в FAISS и генерация ответа

Для каждого подзапроса берём `k` наиболее близких фрагментов из всей базы **и дополнительно** `k_laws`
фрагментов с фильтром `category="laws"` — так текст самой нормы гарантированно попадает в контекст,
а не вытесняется постановлениями и практикой.


In [ ]:
def retrieve(queries, k=4, k_laws=2):
    seen, docs = set(), []
    def add(dlist):
        for d in dlist:
            key = (d.metadata["source"], d.page_content[:120])
            if key not in seen:
                seen.add(key)
                docs.append(d)
    for q in queries:
        add(knowledge_base.similarity_search(q, k=k))
        add(knowledge_base.similarity_search(q, k=k_laws, filter={"category": "laws"}, fetch_k=50))
    return docs

def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        cit = d.metadata.get('citation') or citation_for(d.metadata['source'])
        parts.append(f"[Фрагмент {i}] {cit}\n{d.page_content}")
    return "\n\n".join(parts)

STEP2_SYSTEM = """Ты — нейроюрист. Консультируешь физических лиц ТОЛЬКО по сделкам с ЖИЛОЙ
недвижимостью (купля-продажа, наём, ипотека, ДДУ, дарение, рента, наследование жилья, налоги при
сделках с жильём, государственная регистрация прав).

Правила:
- Отвечай, опираясь ТОЛЬКО на приведённые фрагменты базы знаний и историю диалога.
- Каждый правовой тезис сопровождай ссылкой на конкретную норму с реквизитами. В начале каждого
  фрагмента указан документ-источник — бери реквизиты оттуда:
    • законы — «ст. 551 ГК РФ», «ст. 217.1 НК РФ», «п. 1 ст. 675 ГК РФ»;
    • постановления Пленума — полностью, напр. «Постановление Пленума ВС РФ от 29.05.2012 № 9»;
    • судебная практика — реквизиты конкретного акта, процитированного во фрагменте,
      напр. «Определение Верховного Суда РФ от 27.06.2018 № 305-ЭС18-8320».
- НИКОГДА не пиши «фрагмент N», «пункт 2 фрагмента» — это не ссылка на источник.
- Не выдумывай номера статей и реквизиты. Если нужной нормы во фрагментах нет — просто не ссылайся
  на неё и по возможности ответь на основе того, что есть; не добавляй оговорок вида
  «в фрагментах не указано».
- Если во фрагментах недостаточно данных для ответа по существу — прямо скажи об этом.
- Если вопрос не относится к сделкам с жилой недвижимостью — вежливо откажись консультировать.
- Пиши понятным языком для неюриста, по существу.
- В конце ответа блок «Источники:» с перечнем использованных документов (с реквизитами)."""

def step2_generate(question, docs, history="", problems=None):
    user = (f"Фрагменты базы знаний:\n{format_context(docs)}\n\n"
            f"История диалога:\n{history or '(пусто)'}\n\n"
            f"Вопрос пользователя:\n{question}")
    if problems:
        user += ("\n\nПроверяющий отметил замечания к предыдущему черновику — исправь ТОЛЬКО их, "
                 "сохранив структуру ответа и ссылки на конкретные статьи/акты с реквизитами "
                 "(не заменяй их на «фрагмент N» и не добавляй оговорки-самоопровержения):\n"
                 + "\n".join(f"- {p}" for p in problems))
    return chat(STEP2_SYSTEM, user)

### Шаг 3. Проверка ответа

In [ ]:
STEP3_SYSTEM = """Ты — проверяющий юрист-редактор. Тебе дан ЧЕРНОВИК ответа и ФРАГМЕНТЫ-источники
(у каждого в начале указаны реквизиты документа).

Отметь замечание ТОЛЬКО в этих случаях:
1. В черновике названы статья / закон / постановление, которых НЕТ среди фрагментов —
   ни в тексте, ни в реквизитах документа-источника. Учти: фрагмент с реквизитом
   «ГК РФ, глава 35 (ст. 671–688…)» подтверждает ссылку на любую статью из этого диапазона;
   фрагмент «Судебная практика по ст. 558 ГК РФ» подтверждает ссылку на ст. 558.
2. В черновике есть утверждение о правовом последствии, прямо ПРОТИВОРЕЧАЩЕЕ фрагментам.
3. Ответ выходит за тему сделок с жилой недвижимостью.

НЕ придирайся к форме ссылок и НЕ требуй, чтобы номер статьи буквально встречался в тексте
фрагмента, если фрагмент явно относится к этой норме. Если сомневаешься — считай ссылку допустимой
и замечание НЕ выставляй.

Верни СТРОГО JSON без пояснений:
{"ok": true, "problems": []}   или   {"ok": false, "problems": ["конкретное замечание", "..."]}"""

def step3_verify(draft, docs):
    user = f"ФРАГМЕНТЫ-источники:\n{format_context(docs)}\n\nЧЕРНОВИК ответа:\n{draft}"
    raw = chat(STEP3_SYSTEM, user)
    try:
        data = parse_json_block(raw)
        return bool(data.get("ok")), data.get("problems", [])
    except Exception:
        return True, []          # если проверяющий не вернул JSON — не блокируем ответ

### Оркестратор: связываем три шага + история диалога

In [ ]:
def ask(question, history="", menu_theme=None, verbose=True):
    def log(*a):
        if verbose: print(*a)

    log("=" * 80)
    log("ВОПРОС:", question)
    if menu_theme:
        log("Тема из меню:", menu_theme)

    # --- Шаг 1
    themes, queries = step1_build_queries(question, history, menu_theme)
    log("\n[Шаг 1] Темы:", themes)
    log("[Шаг 1] Поисковые запросы:")
    for q in queries:
        log("   •", q)

    # --- Шаг 2
    docs = retrieve(queries)
    log(f"\n[Шаг 2] Найдено уникальных фрагментов: {len(docs)} из документов:")
    for src, n in Counter(d.metadata.get("citation") or citation_for(d.metadata["source"]) for d in docs).items():
        log(f"   - {src}" + (f"  ×{n}" if n > 1 else ""))
    draft = step2_generate(question, docs, history)
    log("\n[Шаг 2] Черновик ответа:\n" + draft)

    # --- Шаг 3
    ok, problems = step3_verify(draft, docs)
    log("\n[Шаг 3] Проверка:", "OK" if ok else "ЕСТЬ ЗАМЕЧАНИЯ")
    answer = draft
    if not ok:
        for p in problems:
            log("   ! ", p)
        log("\n[Шаг 3] Повторная генерация с учётом замечаний...")
        answer = step2_generate(question, docs, history, problems=problems)
        log("\n[Шаг 3] Исправленный ответ:\n" + answer)

    log("\n" + "-" * 80)
    log("ИТОГОВЫЙ ОТВЕТ:\n" + answer)
    log("=" * 80)
    return answer


class Dialog:
    """Тонкая обёртка для ведения истории диалога."""
    def __init__(self):
        self.history = ""
    def ask(self, question, menu_theme=None, verbose=True):
        answer = ask(question, self.history, menu_theme, verbose)
        self.history += f"Вопрос: {question}\nОтвет: {answer}\n\n"
        return answer

## 5. Демонстрация работы прототипа

Вопросы взяты из тестового набора `testset/test_questions.md` (сформулированы бытовым языком).
Для каждого выводятся все промежуточные шаги алгоритма и итоговый ответ.


### 5.1. Одиночные вопросы по разным темам

In [ ]:
demo_questions = [
    ("купля-продажа", "Подписали договор купли продажи. Квартира уже моя?"),
    ("купля-продажа", "У продавца в свидетельстве он один. Но он женат. Жену его спрашивать надо?"),
    ("наём/аренда",   "Снимаю квартиру. Хозяин ее продал. Мне съезжать?"),
    ("налоги",        "Продаю квартиру. Владею ей два года. Придется платить налог?"),
    ("наследование",  "Отец умер, осталась квартира. К нотариусу не ходил. Но живу там и плачу за коммуналку. Это считается, что я принял наследство?"),
]

for theme, q in demo_questions:
    ask(q, menu_theme=theme)
    print("\n\n")

### 5.2. Составной вопрос (две темы сразу)

Проверяем Шаг 1: вопрос должен разложиться на подвопросы по ДДУ и по ипотеке.


In [ ]:
_ = ask("Купил квартиру по ДДУ в ипотеку. Дом сдали, а там куча недоделок. Что делать?",
        menu_theme="ДДУ")

### 5.3. Диалог с уточняющим вопросом

Второй вопрос не самодостаточен — он понятен только с учётом истории первого.


In [ ]:
d = Dialog()
d.ask("Снимаю квартиру по договору найма на год. Хозяин ее продал. Мне съезжать?",
      menu_theme="наём/аренда")
print("\n\n########## УТОЧНЯЮЩИЙ ВОПРОС ##########\n\n")
_ = d.ask("А новый хозяин может поднять мне плату?")

### 5.4. Вопрос вне периметра

Прототип должен вежливо отказаться, а не консультировать по чужой теме.


In [ ]:
_ = ask("Меня оштрафовали за парковку на газоне. Как обжаловать?")

## 6. Выводы

**Работоспособность.** Прототип отрабатывает сквозной сценарий: RAG на FAISS по базе из 87 документов
(~2700 чанков) + цепочка из трёх вызовов LLM (разбор вопроса → генерация ответа со ссылками →
проверка) + история диалога. Составные вопросы Шаг 1 корректно раскладывает на подвопросы.

**Трудности, выявленные при разработке прототипа** (и как решены):

1. **Юридическая терминология на Шаге 1 определяет качество поиска.** На вопрос «снимаю квартиру,
   хозяин продал» первая версия сформулировала запросы про *аренду*, поиск ушёл в главу 34 ГК, а
   правильная норма (ст. 675 ГК РФ, наём) не попала в контекст. Решение — подсказки по терминологии
   в системном промпте Шага 1 (наём ≠ аренда, момент перехода права — ст. 223/551 ГК и т. д.).

2. **Шаг проверки — палка о двух концах.** Слишком строгий проверяющий браковал верные ссылки
   («ст. 558 нет во фрагментах», хотя фрагмент постатейной практики по ст. 558 был найден) и
   провоцировал повторную генерацию, которая *ухудшала* ответ — заменяла «ст. 558 ГК РФ» на
   «фрагмент 2» и добавляла оговорки «в фрагментах не указано». Решение — смягчить критерии Шага 3
   (реквизиты документа-источника подтверждают ссылку; не придираться к форме) и в промпте повторной
   генерации явно запретить подмену ссылок и самоопровержения.

3. **Ссылки без реквизитов бесполезны для юрконсультации.** Модель писала «постановление Пленума
   ВС РФ» без даты и номера. Решение — справочник реквизитов по каждому файлу базы; реквизиты
   подставляются в контекст и в требования к формату ответа.

4. **Поиск смещён в сторону практики и постановлений.** По ряду вопросов среди top-k фрагментов не
   оказывалось текста самой нормы. Решение — для каждого подзапроса дополнительный поиск с фильтром
   `category="laws"`.

**Открытые вопросы для экспериментальной части (Этап № 4):**
- `chunk_size` / `chunk_overlap`, число фрагментов `k` и `k_laws` — влияние на полноту и точность
  ответа (метрика — доля вопросов из тестового набора, где найден эталонный источник и ответ на
  него корректно ссылается);
- модель эмбеддингов (`text-embedding-3-small` vs `-3-large`);
- версии системных промптов Шага 1 и Шага 3;
- целесообразность Шага 3 в текущем виде и альтернативные схемы проверки.
- База знаний содержит текст статей в действующей редакции, но модель может сослаться на исторически
  изменённую норму (напр. п. 2 ст. 558 ГК РФ о регистрации договора) — нужна работа с метаданными о
  редакции на этапе индексации.

## 7. План дальнейшей работы

1. **Экспериментальная часть (Этап № 4).** Таблица сравнения гиперпараметров на тестовом наборе из
   30 вопросов; зафиксировать итоговую конфигурацию с обоснованием.
2. **Метаданные чанков.** Проставлять при индексации источник, дату проверки и статус редакции
   документа (сейчас ведётся вручную в `MANIFEST.md`); показывать пользователю дату актуальности базы.
3. **Продакшн-интеграция.** Вынести пайплайн в FastAPI-сервис с эндпоинтом `/ask` (разбор, поиск,
   генерация, проверка, история — на бэкенде) и подключить Telegram-бот как тонкий клиент:
   меню тем, текстовый и голосовой ввод (`whisper-1`).
4. **Регламент актуализации базы.** Ежеквартальная сверка редакций законов и новых обзоров практики
   ВС РФ + служебный скрипт полной пересборки индекса FAISS при обновлении файлов базы.
5. **Расширение оценки качества.** Помимо «найден ли эталонный источник» — экспертная проверка
   юридической корректности ответов на репрезентативной выборке.
